In [4]:
import pandas as pd

In [1]:
import os, pickle

In [2]:
with open("../../../training_data/8.Apos/Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pickle.load(f)

len(extras_featuresd), extras_featuresd

(24,
 {'8sgj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       8sgj               1             A           52            A   
  1       8sgj               1             A           53            A   
  2       8sgj               1             A           54            A   
  3       8sgj               1             A           55            A   
  4       8sgj               1             A           56            A   
  ..       ...             ...           ...          ...          ...   
  746     8sgj               1             A          941            A   
  747     8sgj               1             A          942            A   
  748     8sgj               1             A          943            A   
  749     8sgj               1             A          944            A   
  750     8sgj               1             A          945            A   
  
                      

In [5]:
all_extras_seqsd = pd.read_pickle("all_extras_seqsd.pkl")

len(all_extras_seqsd), all_extras_seqsd

(24,
 {('8sgj',
   '1',
   'A'): 'MYNMRRLSLSPTFSMGFHLLVTVSLLFSHVDHVIAETEMEGEGNETGECTGSYYCKKGVILPIWEPQDPSFGDKIARATVYFVAMVYMFLGVSIIADRFMSSIEVITSQEKEITIKKPNGETTKTTVRIWNETVSNLTLMALGSSAPEILLSVIEVCGHNFTAGDLGPSTIVGSAAFNMFIIIALCVYVVPDGETRKIKHLRVFFVTAAWSIFAYTWLYIILSVISPGVVEVWEGLLTFFFFPICVVFAWVADRRLLFYKYVYKRYRAGKQRGMIIEHEGDRPSSKTEIEMDGKVVNSHVENFLDGALVLEVDERDQDDEEARREMARILKELKQKHPDKEIEQLIELANYQVLSQQQKSRAFYRIQATRLMTGATENDPVSKIFFEQGTYQCLENCGTVALTIIRRGGDLTNTVFVDFRTEDGTANAGSDYEFTEGTVVFKPGDTQKEIRVGIIDDDIFEEDENFLVHLSNVKVSSEASEDGILEANHVSTLACLGSPSTATVTIFDDDHAGIFTFEEPVTHVSESIGIMEVKVLRTSGARGNVIVPYKTIEGTARGGGEDFEDTCGELEFQNDEIVKTISVKVIDDEEYEKNKTFFLEIGEPRLVEMSEKKALLLNELGGFTITGKYLFGQPVFRKVHAREHPILSTVITIADEYDDKQPLTSKEEEERRIAEMGRPILGEHTKLEVIIEESYEFKSTVDKLIKKTNLALVVGTNSWREQFIEAITVSAGEDDDDDECGEEKLPSCFDYVMHFLTVFWKVLFAFVPPTEYWNGWACFIVSILMIGLLTAFIGDLASHFGCTIGLKDSVTAVVFVALGTSVPDTFASKVAATQDQYADASIGNVTGSNAVNVFLGIGVAWSIAAIYHAANGEQFKVSPGTLAFSVTLFTIFAFINVGVLLYRRRPEIGGELGGPRTAKLLTSCLFVLLWLLYIFFSSLEAYCHIKGFAAAGGSWSHPQFEKGGGS

In [6]:
extras_seqsd = {}

for pid, seq in all_extras_seqsd.items():
    pdb, entid, chainid = pid
    if pdb in extras_featuresd:
        assert pdb not in extras_seqsd, pdb
        extras_seqsd[pdb] = seq

len(extras_seqsd), extras_seqsd

(24,
 {'8sgj': 'MYNMRRLSLSPTFSMGFHLLVTVSLLFSHVDHVIAETEMEGEGNETGECTGSYYCKKGVILPIWEPQDPSFGDKIARATVYFVAMVYMFLGVSIIADRFMSSIEVITSQEKEITIKKPNGETTKTTVRIWNETVSNLTLMALGSSAPEILLSVIEVCGHNFTAGDLGPSTIVGSAAFNMFIIIALCVYVVPDGETRKIKHLRVFFVTAAWSIFAYTWLYIILSVISPGVVEVWEGLLTFFFFPICVVFAWVADRRLLFYKYVYKRYRAGKQRGMIIEHEGDRPSSKTEIEMDGKVVNSHVENFLDGALVLEVDERDQDDEEARREMARILKELKQKHPDKEIEQLIELANYQVLSQQQKSRAFYRIQATRLMTGATENDPVSKIFFEQGTYQCLENCGTVALTIIRRGGDLTNTVFVDFRTEDGTANAGSDYEFTEGTVVFKPGDTQKEIRVGIIDDDIFEEDENFLVHLSNVKVSSEASEDGILEANHVSTLACLGSPSTATVTIFDDDHAGIFTFEEPVTHVSESIGIMEVKVLRTSGARGNVIVPYKTIEGTARGGGEDFEDTCGELEFQNDEIVKTISVKVIDDEEYEKNKTFFLEIGEPRLVEMSEKKALLLNELGGFTITGKYLFGQPVFRKVHAREHPILSTVITIADEYDDKQPLTSKEEEERRIAEMGRPILGEHTKLEVIIEESYEFKSTVDKLIKKTNLALVVGTNSWREQFIEAITVSAGEDDDDDECGEEKLPSCFDYVMHFLTVFWKVLFAFVPPTEYWNGWACFIVSILMIGLLTAFIGDLASHFGCTIGLKDSVTAVVFVALGTSVPDTFASKVAATQDQYADASIGNVTGSNAVNVFLGIGVAWSIAAIYHAANGEQFKVSPGTLAFSVTLFTIFAFINVGVLLYRRRPEIGGELGGPRTAKLLTSCLFVLLWLLYIFFSSLEAYCHIKGFAAAGGSWSHPQFEKGGGSGGGSGGSAWSHPQFEK',

# Make predictions

deepallo was run in its own environment in /data/fnerin/envs_other_tools, created with the requirements_conda.txt file

changes to inference.py regarding paths and fpocket execution were made

In [7]:
if not os.path.isfile("fpocket"):
    os.system(f"ln -s /home/fnerin/miniconda3/envs/allopockets/bin/fpocket fpocket") # os.getcwd().rsplit('/', 3)[0] --> absolute path of allodb_new

In [8]:
from pathlib import Path
from Bio.SeqUtils import seq3

In [9]:
for pdb, feats in extras_featuresd.items():
    if not os.path.isfile(f"{pdb}/{pdb}.pkl"):
        os.makedirs(pdb, exist_ok=True)
    
        # The script is hardcoded for rcsb-downloaded .pdb files; some PDBs do not have .pdb available anymore, so the SEQRES is reconstructed below
        # if not os.path.isfile(f"{pdb}/{pdb}.pdb"):
        #     os.system(f"ln -s {os.getcwd().rsplit('/', 1)[0]}/structures/{pdb}.pdb {pdb}/{pdb}.pdb")
        #     # os.getcwd().rsplit('/', 1)[0] --> absolute path of previous directory
        chains = feats[('Residues', 'auth_asym_id')].unique()
        if len(chains) == 1:
            chain_id = chains.item()

            # SEQRES in RCSB .pdb coincides with the pdbx-deposited sequence (canonical) in RCSB .cif
            seq = extras_seqsd[pdb]
            seqres = [
                f"SEQRES {i//13 + 1:>3} {chain_id} {len(seq):>4}  "
                + " ".join(seq3(aa).upper() for aa in seq[i:i+13])
                + "\n"
                for i in range(0, len(seq), 13)
            ]

            with open(f"{pdb}/{pdb}.pdb", "w") as out, open(f"../structures/{pdb}.pdb") as inp:
                out.writelines(seqres)
                out.writelines(inp)

            
            os.system(f"python deepallo/inference.py {pdb} {chain_id}") # If it is not a single chain, .item() will fail
        else:
            assert False
            # for chain in chains:
            #     if os.path.isdir(f"{pdb}_{chain}"): continue
            #     os.makedirs(pdb, exist_ok=True)
            #     os.system(f"python deepallo/inference.py {pdb} {chain}") # If it is not a single chain, .item() will fail
            #     os.system(f"mv {pdb} {pdb}_{chain}")

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/huggingface_hub/file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 328 files:  33%|███▎      | 108/328 [00:00<00:00, 1074.57it/s]

Fetching 328 files: 100%|██████████| 328/328 [00:00<00:00, 1204.00it/s]


***** POCKET HUNTING BEGINS ***** 


***** POCKET HUNTING ENDS ***** 


mv: '6z1h/6z1h_out' and '6z1h/6z1h_out' are the same file


...

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from

/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/data/fnerin/envs_other_tools/deepallo/lib/python3.11/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.0 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


(0.05952753871679306, ['701', '459', '704', '462', '467', '463'])
(0.04623870551586151, ['179', '176', '296', '667', '215', '297', '461', '693', '663', '183', '670', '454', '300', '694', '270', '690', '182', '277', '273', '666'])
(0.045989617705345154, ['645', '654', '647', '641', '651', '650'])


## AlphaFold model

- Needs the SEQRES in a .pdb; AT LEAST
- Edited utils.sequence_indices to effectively get the .cif from EBI.

## 8vw5

Does not have a legacy .pdb to download.

In [10]:
# Any missing
for pdb, feats in extras_featuresd.items():
    if not os.path.isfile(f"{pdb}/{pdb}.pkl"):
        print("Missing:", pdb)

# Processing

In [11]:
import pandas as pd

In [12]:
results = {
    pdb: {
        pocket: {
            "prob": pocketd["prob"],
            "residues": pd.DataFrame(pocketd["res"])
        }
        for pocket, pocketd in pd.read_pickle(f"{pdb}/{pdb}.pkl").items()
    }
    for pdb in extras_featuresd
    # if pdb != "7sns"
}

len(results), results

(24,
 {'8sgj': {0: {'prob': 0.06474058330059052,
    'residues':    auth_asym_id auth_seq_id
    0             A         172
    1             A         827
    2             A          98
    3             A         837
    4             A         210
    5             A         830
    6             A         213
    7             A         103
    8             A         165
    9             A         825
    10            A         102
    11            A         224
    12            A         168
    13            A         244
    14            A         826
    15            A          99
    16            A         840
    17            A         222
    18            A         171
    19            A         833
    20            A         214
    21            A         209
    22            A         836
    23            A          97
    24            A         175
    25            A         829},
   1: {'prob': 0.0645398423075676,
    'residues':    auth_asym_id auth_s

In [13]:
from Bio import PDB

def process_pocket(f):
    s = PDB.PDBParser(QUIET=True).get_structure("protein", f)
    return pd.DataFrame((
        {
            "auth_asym_id": chain.id,
            "auth_seq_id": res.get_id()[1],
            "pdbx_PDB_ins_code": res.get_id()[2] if res.get_id()[2] != " " else "?"
        }
        for model in s
        for chain in model
        for res in chain
    ), dtype=str).drop_duplicates()

In [14]:
processed_results = {}

for pdb in extras_featuresd:
    # if pdb == "7sns": continue
    processed_results[pdb] = {}
    
    pdbpath = f"{pdb}/{pdb}_out/pockets"
    for pocketf in os.listdir(pdbpath):
        if pocketf.endswith("_atm.pdb"):
            pocketn = pocketf.replace("_atm.pdb", "")
            res = process_pocket(f"{pdbpath}/{pocketf}")

            d = {"pred": 0, "residues": res}
            
            existing = tuple(
                pocket
                for pocket, pocketd in results[pdb].items()
                if len( pocketd["residues"].merge(res[["auth_asym_id", "auth_seq_id"]]) ) == len( pocketd["residues"] )
            )
            if len(existing):
                d.update({"pred": 1, "prob": results[pdb][existing[0]]["prob"]})
            elif len(existing) >= 2:
                raise Exception("More than 1 matching + pred pocket" + pdb + pocketn)

            processed_results[pdb][pocketn] = d
    
    assert sum(v["pred"] for v in processed_results[pdb].values()) == 3, "Not all + pred pockets were picked up" + pdb
    processed_results[pdb] = dict(sorted(
        processed_results[pdb].items(), 
        key = lambda x: (
            x[-1].get("prob", 0), 
            1/int(x[0].replace("pocket", ""))
        ), 
        reverse = True
    ))
    
len(processed_results), processed_results

(24,
 {'8sgj': {'pocket1': {'pred': 1,
    'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
    0             A         175                 ?
    1             A         171                 ?
    2             A         172                 ?
    3             A         210                 ?
    4             A         209                 ?
    5             A         840                 ?
    6             A         833                 ?
    7             A          99                 ?
    8             A         168                 ?
    9             A         213                 ?
    10            A         103                 ?
    11            A         214                 ?
    12            A         244                 ?
    13            A         825                 ?
    14            A         827                 ?
    15            A         837                 ?
    16            A         836                 ?
    17            A         826                 ?

In [15]:
resultsf = "deepallo_results.pkl"

with open(resultsf, "wb") as f:
    pickle.dump(processed_results, f)